# 🛡️ ROAD-SHIELD: Full Deep Training Pipeline (Google Colab GPU)

**SIH 2026 Problem Statement SIH26124 | Bharat Electronics Limited (BEL) | Team 42**

This notebook runs the **complete end-to-end deep training pipeline** on a Colab GPU (T4/A100).

### Training Stages (~2-3 hours on T4 GPU):
| Stage | Component | Estimated Time |
|-------|-----------|----------------|
| 1 | Environment Setup & Dataset Fetch | ~5 min |
| 2 | Civil Engineering Models (PCI, Depth, Deterioration) | ~2 min |
| 3 | Deep 1-D CNN IMU Shock Classifier | ~10 min |
| 4 | Deep U-Net Pixel Defect Segmenter (ResNet-34 encoder) | ~45-60 min |
| 5 | End-to-End CNN Fine-Tuning (EfficientNet, MobileNet, ResNet) | ~60-75 min |
| 6 | Flagship DAN-DAG (Dual Attention + Decision DAG) | ~15 min |
| 7 | Model Selection & Full Test Suite Verification | ~5 min |

### Instructions:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. When prompted, authorize Google Drive (to save checkpoints)
4. Wait ~2-3 hours for training to complete
5. Trained checkpoints zip will be saved to your Google Drive

---
## Cell 1: Mount Google Drive & Clone Repository

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository
!git clone -b add-dan-dag https://github.com/udbhav968-creator/SIH_PROJECT.git /content/SIH_PROJECT 2>/dev/null || \
    (cd /content/SIH_PROJECT && git pull origin add-dan-dag)

%cd /content/SIH_PROJECT
!echo "Repository ready at $(pwd)"

---
## Cell 2: Install Dependencies & Verify GPU

In [ ]:
# Install required Python packages
!pip install -q "scikit-learn>=1.8.0,<1.9" "opencv-python-headless>=4.8,<5" \
    onnxruntime-gpu onnx onnxscript scikit-image joblib pillow scipy

# Verify environment
import torch
import sklearn
import cv2
import platform

print(f"Python {platform.python_version()} | sklearn {sklearn.__version__} | cv2 {cv2.__version__}")
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

# Set environment
import os
os.environ['PYTHONPATH'] = '/content/SIH_PROJECT'
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

---
## Cell 3: Fetch & Expand Datasets (Rare Class Augmentation)

In [ ]:
%cd /content/SIH_PROJECT

# Fetch main crack/pothole dataset (DNIT crops)
!python -m scripts.fetch_cracks_potholes_dataset --limit 2235 --workers 16 2>&1 | tail -8

# Expand rare classes (Waterlogging, Zebra, Divider, Damaged Signs)
!python -m scripts.expand_rare_classes --target 80 2>&1 | tail -10

# Show corpus inventory
!python -c "
from data.image_dataset import CLASS_FOLDERS
import os
print('Class Distribution:')
print('-' * 55)
total = 0
for cid, (folder, name) in sorted(CLASS_FOLDERS.items()):
    d = os.path.join('datasets', folder, 'real_images')
    n = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(d) else 0
    total += n
    print(f'  [{cid}] {name:45s} {n:5d} images')
print('-' * 55)
print(f'  Total: {total} images')
"

---
## Cell 4: Train Civil Engineering Models (PCI, Deterioration, Depth)

In [ ]:
%cd /content/SIH_PROJECT
!python -m training.train_civil_real

---
## Cell 5: Train Deep 1-D CNN IMU Shock Classifier (~10 min)

In [ ]:
%cd /content/SIH_PROJECT
!python -m training.train_imu_deep 2>&1 | tee logs/imu_deep.txt | tail -30

---
## Cell 6: Train Deep U-Net Pixel Defect Segmenter (~45-60 min GPU)

This trains a U-Net with a ResNet-34 encoder for pixel-level crack/pothole segmentation on GPU.

In [ ]:
%cd /content/SIH_PROJECT
!python -m training.train_unet_segmenter 2>&1 | tee logs/unet_seg.txt

---
## Cell 7: End-to-End CNN Fine-Tuning on GPU (~60-75 min)

Fine-tunes EfficientNet-B0, EfficientNet-B2, MobileNetV3-Large, and ResNet-50 with unfrozen convolutions.
Each architecture trains for 40 epochs with early stopping.

In [ ]:
%cd /content/SIH_PROJECT

ARCHS = "efficientnet_b0,efficientnet_b2,mobilenet_v3_large,resnet50"
EPOCHS = 40
BATCH = 48

# Train each architecture one at a time (GPU memory friendly)
for arch in ARCHS.split(','):
    print(f"\n{'='*70}")
    print(f"  FINE-TUNING: {arch} ({EPOCHS} epochs, batch {BATCH})")
    print(f"{'='*70}")
    !python -m training.train_finetune_cnn --archs {arch} --epochs {EPOCHS} \
        --batch {BATCH} --workers 2 --resume --train-only 2>&1 | \
        grep -E "VAL|early stop|reusing|epoch +(1|10|20|30|40)/|Error|error"

# Final model selection: refit best architecture on train+val
print(f"\n{'='*70}")
print("  MODEL SELECTION & REFIT ON FULL TRAIN+VAL")
print(f"{'='*70}")
!python -m training.train_finetune_cnn --archs {ARCHS} --epochs {EPOCHS} \
    --batch {BATCH} --workers 2 --resume 2>&1 | \
    grep -E "VAL|refit|reusing|chosen|served|Error|error"

---
## Cell 8: Flagship DAN-DAG Network (Dual Attention + Decision DAG, ~15 min)

In [ ]:
%cd /content/SIH_PROJECT
!python -m training.train_dan_dag --backbone mobilenetv2 --epochs 65

---
## Cell 9: CNN Embedding Head (Frozen Transfer Learning Baseline)

In [ ]:
%cd /content/SIH_PROJECT
!python -m training.train_cnn_head --backbone mobilenetv2 --compare 2>&1 | \
    grep -v "^    embedded" | tee logs/cnn_head.txt | tail -30

---
## Cell 10: Model Selection & Verification Test Suite

In [ ]:
%cd /content/SIH_PROJECT

# Select the best vision model
!python -m scripts.select_vision_model 2>&1 || true

# Run full regression test suite
print("\n" + "="*70)
print("  RUNNING FULL VERIFICATION TEST SUITE")
print("="*70)
!python -m unittest discover -s tests -t . 2>&1 | tail -5

---
## Cell 11: Package & Save Checkpoints to Google Drive

In [ ]:
%cd /content/SIH_PROJECT
import os, shutil

# Package all trained checkpoints
!zip -q -r /content/road_shield_trained_checkpoints.zip checkpoints/ logs/

# Copy to Google Drive
drive_dir = '/content/drive/MyDrive/ROAD_SHIELD_Training'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy('/content/road_shield_trained_checkpoints.zip', drive_dir)

# Show summary
zip_size = os.path.getsize('/content/road_shield_trained_checkpoints.zip') / 1e6
print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"  Checkpoints zip: {zip_size:.1f} MB")
print(f"  Saved to Google Drive: {drive_dir}/road_shield_trained_checkpoints.zip")
print(f"{'='*70}")

# List all trained model files
print("\nTrained Checkpoints:")
for f in sorted(os.listdir('checkpoints')):
    fp = os.path.join('checkpoints', f)
    if os.path.isfile(fp):
        sz = os.path.getsize(fp) / 1e6
        print(f"  {f:50s} {sz:7.2f} MB")